# Euler's Method and Order of Accuracy: Class Notebook
MATH-3515 Numerical Analysis · Module 13, Class 1

This notebook goes with the slides *Euler's Method and Order of Accuracy*. Each part starts where a slide says **→ Notebook, Part N**.

All the code is given. Run the cells in order (Runtime → Run all works too) and read the output. Each part ends with **What to notice**.

### Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(precision=6)
# Okabe-Ito colorblind-safe palette
OI = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]

# The test problem: logistic model x' = r x (1 - x/K), x(0) = x0, up to time T
r, K, x0, T = 0.5, 700.0, 30.0, 10.0

def f(t, x):
    return r * x * (1 - x / K)

def exact(t):
    return K * x0 / (x0 + (K - x0) * np.exp(-r * t))

def n_steps(T, h):
    n = round(T / h)
    if abs(n * h - T) > 1e-12:
        raise ValueError("T/h must be a whole number; choose h = 2**-k")
    return n

print(f"Ready. Exact x(10) = {exact(T):.10f}")

## Part 1. Euler's method on the logistic model
*In class: run the three code cells of this part and compare with the hand work.*

Euler's method follows the tangent line for a time $h$:
$$x_{k+1} = x_k + h\, f(t_k, x_k).$$

The first function is the code from the slide *Euler in Six Lines*, with one change: the slide counts `n` steps, and here you give the final time `T` and `n_steps` finds `n`. The second function does the same work and keeps the whole path.

In [ ]:
def euler(f, t0, x0, h, T):
    n = n_steps(T - t0, h)
    t, x = t0, x0
    for k in range(n):
        x = x + h * f(t, x)     # follow the tangent line
        t = t + h
    return x

def euler_path(f, t0, x0, h, T):
    n = n_steps(T - t0, h)
    t = t0 + h * np.arange(n + 1)
    x = np.zeros(n + 1)
    x[0] = x0
    for k in range(n):
        x[k + 1] = x[k] + h * f(t[k], x[k])
    return t, x

In [ ]:
# Hand check (slide "Two Steps by Hand"): h = 1, two steps
hand_t, hand_x = euler_path(f, 0.0, x0, 1.0, 2.0)

print(f"{'t':>3s} {'Euler':>9s} {'exact':>9s} {'error':>9s}")
for t_k, x_k in zip(hand_t[1:], hand_x[1:]):
    print(f"{t_k:>3.0f} {x_k:>9.3f} {exact(t_k):>9.3f} {abs(exact(t_k) - x_k):>9.3f}")

**Predict.** On one plot: the exact curve and Euler with $h$ = 2, 1, 1/2. Which $h$ ends closest to $x(10)$? Is Euler above or below the curve?

In [ ]:
fine_t = np.linspace(0, T, 400)
final_values = {}

plt.figure(figsize=(7, 4.5))
plt.plot(fine_t, exact(fine_t), "-", color=OI[6], linewidth=2, label="exact solution")
for h, name, marker, style, color in ((2.0, "2", "o", "--", OI[1]),
                                      (1.0, "1", "s", "-.", OI[0]),
                                      (0.5, "1/2", "^", ":", OI[2])):
    t_path, x_path = euler_path(f, 0.0, x0, h, T)
    plt.plot(t_path, x_path, linestyle=style, marker=marker, color=color, label=f"Euler, h = {name}")
    final_values[h] = x_path[-1]
    print(f"h = {name:>3s}:  x_n = {x_path[-1]:.4f},  error = {abs(exact(T) - x_path[-1]):.4f}")
plt.axhline(K, linestyle="--", color=OI[3], label="carrying capacity K = 700")
plt.xlabel("time t")
plt.ylabel("population x(t)")
plt.title("Logistic model: exact solution and Euler's method")
plt.grid(alpha=0.3)
plt.legend(loc="best", fontsize=8)
plt.show()

**What to notice.**
- Euler stays **below** the curve here: the curve bends upward early, so the tangent line undershoots, and the deficit is carried forward.
- The error at $T = 10$ drops as $h$ drops: 80.7687 for $h = 2$, 27.9030 for $h = 1$, 11.7637 for $h = 1/2$.
- Two steps by hand already show errors accumulating: 3.8, then 10.8.

## Part 2. Order of accuracy: halve $h$, halve the error
*In class: run the table and the log-log plot. The Try it cell is for after class.*

**Predict.** If $h$ is halved, what happens to Euler's error at $T = 10$? (a) about the same, (b) about half, (c) about a quarter.

In [ ]:
def h_label(k):
    return "1" if k == 0 else f"1/{2**k}"

def err_text(e):                    # rounded as on the slide
    return f"{e:#.4g}" if e >= 1 else f"{e:.3f}"

h_values = np.array([2.0 ** -k for k in range(7)])
x_values = np.zeros(7)
err_values = np.zeros(7)

print(f"{'h':>5s} {'n':>5s} {'Euler x_n':>10s} {'error':>7s} {'ratio':>6s}")
for i, h in enumerate(h_values):
    n = n_steps(T, h)
    x_values[i] = euler(f, 0.0, x0, h, T)
    err_values[i] = abs(exact(T) - x_values[i])
    ratio = "" if i == 0 else f"{err_values[i - 1] / err_values[i]:.2f}"
    print(f"{h_label(i):>5s} {n:>5d} {x_values[i]:>10.3f} {err_text(err_values[i]):>7s} {ratio:>6s}")
ratios = err_values[:-1] / err_values[1:]

In [ ]:
slope_all = np.polyfit(np.log(h_values), np.log(err_values), 1)[0]
slope_last4 = np.polyfit(np.log(h_values[-4:]), np.log(err_values[-4:]), 1)[0]
print(f"Fitted slope, all points:   {slope_all:.3f}")
print(f"Fitted slope, last 4 points: {slope_last4:.3f}")

plt.figure(figsize=(6.5, 4.5))
plt.loglog(h_values, err_values, "o-", color=OI[0], label="Euler")
plt.loglog(h_values, err_values[-1] * (h_values / h_values[-1]), "--", color=OI[6], label="slope 1")
plt.xlabel("step size h")
plt.ylabel("error at T = 10")
plt.title("Euler's method: error against step size")
plt.grid(alpha=0.3, which="both")
plt.legend(loc="best", fontsize=8)
plt.show()

**Try it: edit and run.** Rerun the table for another final time. Try `T_try = 4` and `T_try = 20`. `T_try` must be a whole number, so that $T/h$ is a whole number for every $h = 2^{-k}$.

In [ ]:
# Try it: edit and run
T_try = 10
k_list = range(0, 7)

previous = None
print(f"{'h':>5s} {'n':>5s} {'Euler x_n':>10s} {'error':>9s} {'ratio':>6s}")
for k in k_list:
    h_try = 2.0 ** -k
    x_try = euler(f, 0.0, x0, h_try, T_try)
    err_try = abs(exact(T_try) - x_try)
    ratio_try = "" if previous is None else f"{previous / err_try:.2f}"
    print(f"{h_label(k):>5s} {n_steps(T_try, h_try):>5d} {x_try:>10.3f} {err_try:>9.4g} {ratio_try:>6s}")
    previous = err_try

**What to notice.**
- The ratio approaches 2: Euler is **first order**.
- On log-log axes the slope is the order $p$.
- The first points are not yet in the "small $h$" regime, which is why the fit over all points (1.066) is a bit above the fit over the last 4 (1.015).
- Each extra correct digit costs 10 times more steps.

## Part 3. Runge-Kutta (RK4) and `solve_ivp`
*In class: run the `rk4` cell, the table, and the `solve_ivp` cell (the last code cell of this part). The log-log plot and the same-cost table can wait until after class.*

RK4 samples the slope four times per step, with the formulas on the slide *Better Slopes*:
$$x_{k+1} = x_k + \frac{h}{6}\,(k_1 + 2k_2 + 2k_3 + k_4).$$

In [ ]:
def rk4(f, t0, x0, h, T):
    n = n_steps(T - t0, h)
    t, x = t0, x0
    for k in range(n):
        k1 = f(t, x)
        k2 = f(t + h / 2, x + h / 2 * k1)
        k3 = f(t + h / 2, x + h / 2 * k2)
        k4 = f(t + h, x + h * k3)
        x = x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t = t + h
    return x

In [ ]:
table_h = np.array([2.0 ** -k for k in range(7)])
table_euler = np.array([abs(exact(T) - euler(f, 0.0, x0, h, T)) for h in table_h])
table_rk4 = np.array([abs(exact(T) - rk4(f, 0.0, x0, h, T)) for h in table_h])
ratio_euler = table_euler[:-1] / table_euler[1:]
ratio_rk4 = table_rk4[:-1] / table_rk4[1:]

print(f"{'h':>5s} {'Euler error':>12s} {'ratio':>6s} {'RK4 error':>11s} {'ratio':>6s}")
for i in range(6):
    r_e = "" if i == 0 else f"{ratio_euler[i - 1]:.2f}"
    r_r = "" if i == 0 else f"{ratio_rk4[i - 1]:.2f}"
    print(f"{h_label(i):>5s} {err_text(table_euler[i]):>12s} {r_e:>6s} {table_rk4[i]:>11.3e} {r_r:>6s}")

slope_rk4 = np.polyfit(np.log(table_h[-4:]), np.log(table_rk4[-4:]), 1)[0]
print(f"Fitted RK4 slope over h = 1/8, 1/16, 1/32, 1/64: {slope_rk4:.2f}")

In [ ]:
plt.figure(figsize=(6.5, 4.5))
plt.loglog(table_h, table_euler, "o-", color=OI[0], label="Euler")
plt.loglog(table_h, table_rk4, "s--", color=OI[1], label="RK4")
plt.loglog(table_h, table_euler[-1] * (table_h / table_h[-1]), ":", color=OI[6], label="slope 1")
plt.loglog(table_h, table_rk4[-1] * (table_h / table_h[-1]) ** 4, "-.", color=OI[3], label="slope 4")
plt.xlabel("step size h")
plt.ylabel("error at T = 10")
plt.title("Euler and RK4: error against step size")
plt.grid(alpha=0.3, which="both")
plt.legend(loc="best", fontsize=8)
plt.show()

In [ ]:
# Same cost: count the evaluations of f
calls = 0
def f_counted(t, x):
    global calls
    calls += 1
    return f(t, x)

same_cost = []
print(f"{'method':>6s} {'evaluations':>12s} {'h':>5s} {'error':>10s}")
for name, method, k in (("Euler", euler, 2), ("RK4", rk4, 0), ("Euler", euler, 4), ("RK4", rk4, 2)):
    calls = 0
    x_end = method(f_counted, 0.0, x0, 2.0 ** -k, T)
    error = abs(exact(T) - x_end)
    same_cost.append((name, calls, 2.0 ** -k, error))
    print(f"{name:>6s} {calls:>12d} {h_label(k):>5s} {error:>10.4g}")

In [ ]:
runs = (("RK45 defaults", {}),
        ("RK45, rtol 1e-6, atol 1e-9", dict(rtol=1e-6, atol=1e-9)),
        ("RK45, rtol 1e-10, atol 1e-10", dict(rtol=1e-10, atol=1e-10)),
        ("LSODA defaults", dict(method="LSODA")))

ivp_results = {}
print(f"{'run':>30s} {'error at T':>11s} {'nfev':>5s} {'steps':>6s}")
for name, options in runs:
    sol = solve_ivp(f, (0, T), [x0], **options)
    error = abs(exact(T) - sol.y[0, -1])
    ivp_results[name] = (error, sol.nfev, len(sol.t) - 1)
    print(f"{name:>30s} {error:>11.3e} {sol.nfev:>5d} {len(sol.t) - 1:>6d}")

**What to notice.**
- The RK4 ratios approach 16: **order 4**.
- For the same work, RK4 is about 100 to 5000 times more accurate here.
- `solve_ivp` with default tolerances is off by about 0.18 at $T = 10$: fine for a plot, not for a table of digits. **Tighten the tolerances and check on a test problem** before trusting it.
- The number of steps `solve_ivp` takes is its own choice (adaptive).

## Part 4 (If time). The rounding floor
*If time — the slides show the result; run it after class if we skip it.*

The total error is about
$$C h^p + \frac{T}{h}\, c\, \epsilon_{\text{mach}}.$$
The first term falls as $h$ shrinks; the second grows. Two experiments show the second term:

- Euler in **single precision** (`np.float32`, $\epsilon \approx$ 1.19e-7), with every quantity cast to `np.float32`, including $h$, $r$, $K$ and the constant 1.
- RK4 in double precision.

The next cell takes a few seconds: the last Euler run has about 10 million steps.

In [ ]:
def euler32(h):
    h = np.float32(h)
    r32, K32, one = np.float32(r), np.float32(K), np.float32(1)
    x = np.float32(x0)
    for _ in range(n_steps(T, float(h))):
        x = x + h * (r32 * x * (one - x / K32))
    return x

floor_k = list(range(2, 21, 2))
floor_e32, floor_e64, floor_rk4 = {}, {}, {}

print(f"{'k':>3s} {'h':>9s} {'steps':>9s} {'Euler float32':>14s} {'Euler float64':>14s} {'RK4 float64':>12s}")
for k in floor_k:
    h = 2.0 ** -k
    floor_e32[k] = abs(exact(T) - float(euler32(h)))
    text64, textrk = "-", "-"
    if k <= 16:
        floor_e64[k] = abs(exact(T) - euler(f, 0.0, x0, h, T))
        floor_rk4[k] = abs(exact(T) - rk4(f, 0.0, x0, h, T))
        text64, textrk = f"{floor_e64[k]:.3e}", f"{floor_rk4[k]:.3e}"
    print(f"{k:>3d} {h:>9.2e} {n_steps(T, h):>9d} {floor_e32[k]:>14.3e} {text64:>14s} {textrk:>12s}")

best32 = min(floor_e32, key=floor_e32.get)
bestrk = min(floor_rk4, key=floor_rk4.get)
print(f"Smallest Euler float32 error: {floor_e32[best32]:.3e} at h = {2.0 ** -best32:.2e}")
print(f"Smallest RK4 float64 error:   {floor_rk4[bestrk]:.3e} at h = {2.0 ** -bestrk:.2e}")

In [ ]:
h32 = np.array([2.0 ** -k for k in floor_e32])
hrk = np.array([2.0 ** -k for k in floor_rk4])

plt.figure(figsize=(7, 5))
plt.loglog(h32, list(floor_e32.values()), "o-", color=OI[0], label="Euler, float32")
plt.loglog(hrk, list(floor_rk4.values()), "s--", color=OI[1], label="RK4, float64")
plt.loglog(2.0 ** -best32, floor_e32[best32], "*", color=OI[6], markersize=14, label="smallest error, Euler float32")
plt.loglog(2.0 ** -bestrk, floor_rk4[bestrk], "D", color=OI[6], markersize=8, label="smallest error, RK4 float64")
plt.annotate(f"{floor_e32[best32]:.1e}", (2.0 ** -best32, floor_e32[best32]),
             textcoords="offset points", xytext=(8, 10))
plt.annotate(f"{floor_rk4[bestrk]:.1e}", (2.0 ** -bestrk, floor_rk4[bestrk]),
             textcoords="offset points", xytext=(8, 10))
plt.xlabel("step size h")
plt.ylabel("error at T = 10")
plt.title("The rounding floor: smaller h is not always better")
plt.grid(alpha=0.3, which="both")
plt.legend(loc="best", fontsize=8)
plt.show()

**What to notice.**
- Each curve falls (method error), reaches a floor, and **rises** (rounding error, one per step).
- The best $h$ for float32 Euler is near 2e-4, with error about 4e-3: about four or five correct digits out of a possible seven.
- RK4 in float64 bottoms out near 7e-13, and the floor is flat for $h$ between about 1e-3 and 2e-4. At $x$ near 608, adjacent doubles are 1.1e-13 apart, so that is only about 6 units in the last place: rounding has used up nearly all the digits.
- **Smaller $h$ is not always better** (Module 2: every operation rounds).

## Preview of Thursday
Euler with $h$ = 3, 4.5, 5.5 ($hr$ = 1.5, 2.25, 2.75) for 40 steps. The cell prints the last four values.

In [ ]:
preview = {}
for h in (3.0, 4.5, 5.5):
    x_path = euler_path(f, 0.0, x0, h, 40 * h)[1]
    preview[h] = x_path[-4:]
    print(f"h = {h}, hr = {h * r}:", ", ".join(f"{v:.1f}" for v in x_path[-4:]))

The true solution never oscillates and never passes $K = 700$. With a large $h$, Euler does both. Thursday: why, and how to predict the largest safe $h$.

## Summary

| Part | Slide | Takeaway |
|---|---|---|
| 1 | Euler in Six Lines | Euler follows the tangent line, and each step starts from the last step's error. |
| 2 | Halve $h$, Halve the Error; Reading a Log-Log Plot | Euler has order 1: the ratio approaches 2 and the log-log slope is 1. |
| 3 | Better Slopes to The Professional Tool | RK4 has order 4 and wins at the same cost; check `solve_ivp` on a test problem. |
| 4 | Can $h$ Go to Zero; Two Floors | Method error plus rounding error gives a best $h$. |

## Confidence check
Rate each statement from 1 (not yet) to 4 (confident).

- I can take an Euler step by hand.
- I can read the order of a method from a table or a log-log plot.
- I can explain why RK4 is worth four evaluations per step.
- I can explain why a smaller $h$ can give a larger error.